# AGEF-Net: lean revision notebook (low memory)
Rebuilds only the CTU-13 part and runs the revision experiments. Safe to use **Runtime -> Run all**. Do NOT load the old checkpoint (it is corrupted and not needed).

In [1]:
!pip install -q geoip2 lightgbm xgboost scikit-learn pandas numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 1.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import geoip2.database, ipaddress
GEOLITE_DB_PATH = '/content/drive/MyDrive/CICIDS2017/GeoLite2-City.mmdb'   # <- change if your file is elsewhere
geo_reader = geoip2.database.Reader(GEOLITE_DB_PATH)
def get_country(ip):
    try:
        ip_obj = ipaddress.ip_address(ip)
        if ip_obj.is_private:
            return 'Not Geolocatable (Private/Internal IP)'
        response = geo_reader.city(ip)
        return response.country.name or 'Unknown'
    except Exception:
        return 'Not Geolocatable (Invalid IP)'
print(get_country('8.8.8.8')); print(get_country('192.168.1.5'))

United States
Not Geolocatable (Private/Internal IP)


## Rebuild CTU-13 data + base models (about 5-10 min incl. download)

In [4]:
import os, gc, subprocess, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
DATA_DIR = '/content'          # CTU-13 files are downloaded here (re-downloaded if missing)

scenarios = {
    42: "https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-42/capture20110810.binetflow.2format",
    43: "https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-43/capture20110811.binetflow.2format",
    44: "https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-44/capture20110812.binetflow.2format",
}
cols = ['saddr','daddr','proto','sport','dport','state','stos','dtos','swin','dwin','shops','dhops',
        'stime','ltime','sttl','dttl','tcprtt','synack','ackdat','spkts','dpkts','sbytes','dbytes',
        'sappbytes','dappbytes','dur','pkts','bytes','appbytes','rate','srate','drate','label']
keep_cols = ['saddr','daddr','proto','state','sttl','dttl','spkts','dpkts','sbytes','dbytes',
             'dur','rate','srate','drate','label']

# ---- 1) load (same 30% chunk sampling and seed as the original cell) ----
all_dfs = []
for sid, url in scenarios.items():
    fname = f'{DATA_DIR}/ctu13_scenario{sid}.binetflow'
    if not os.path.exists(fname):
        print(f'Downloading scenario {sid} ...')
        subprocess.run(['wget', '-q', '-O', fname, url])
    parts = []
    for chunk in pd.read_csv(fname, names=cols, usecols=keep_cols, low_memory=False, chunksize=200000):
        chunk = chunk[chunk['saddr'] != 'SrcAddr']
        parts.append(chunk.sample(frac=0.3, random_state=42) if len(chunk) > 50000 else chunk)
    d = pd.concat(parts, ignore_index=True); d['scenario'] = sid
    all_dfs.append(d); print(f'  scenario {sid}: {len(d)} rows kept'); del parts; gc.collect()
df_multi = pd.concat(all_dfs, ignore_index=True); del all_dfs; gc.collect()
df_multi['Label'] = df_multi['label'].astype(str).str.lower().apply(lambda x: 1 if 'botnet' in x else 0)

# ---- 2) features (identical to the original cell) ----
feature_cols = ['dur','spkts','dpkts','sbytes','dbytes','rate','srate','drate','sttl','dttl']
df_model = df_multi.copy()
for c in feature_cols:
    df_model[c] = pd.to_numeric(df_model[c], errors='coerce')
df_model = df_model.dropna(subset=feature_cols)

top_states = df_model['state'].value_counts().head(10).index
df_model['state_grouped'] = df_model['state'].where(df_model['state'].isin(top_states), 'Other')
top_protos = df_model['proto'].value_counts().head(5).index
df_model['proto_grouped'] = df_model['proto'].where(df_model['proto'].isin(top_protos), 'Other')
df_model = pd.concat([df_model, pd.get_dummies(df_model['proto_grouped'], prefix='proto'),
                      pd.get_dummies(df_model['state_grouped'], prefix='state')], axis=1)
feature_cols_final = feature_cols + [c for c in df_model.columns
    if (c.startswith('proto_') or c.startswith('state_')) and c not in ['proto_grouped', 'state_grouped']]

# ---- 3) data-attrition audit (this replaces cell R1) ----
audit = pd.DataFrame({
    'rows_loaded': df_multi.groupby('scenario').size(),
    'attacks_loaded': df_multi.groupby('scenario')['Label'].sum(),
    'rows_modelled': df_model.groupby('scenario').size(),
    'attacks_modelled': df_model.groupby('scenario')['Label'].sum()}).fillna(0).astype(int)
audit['attacks_dropped'] = audit['attacks_loaded'] - audit['attacks_modelled']
audit['pct_attacks_dropped'] = (100 * audit['attacks_dropped'] / audit['attacks_loaded']).round(1)
print('\n=== DATA ATTRITION AUDIT ==='); print(audit)
audit.to_csv('table_data_attrition_audit.csv')
tmp = df_multi[feature_cols].apply(pd.to_numeric, errors='coerce')
lost = tmp[tmp.isna().any(axis=1) & (df_multi['Label'] == 1)]
print('\nMissing values in the dropped ATTACK rows, by column:'); print(lost.isna().sum())
del tmp, lost, df_multi; gc.collect()

# ---- 4) same stratified split as the paper, then free memory ----
X_ctu = df_model[feature_cols_final]; y_ctu = df_model['Label']
X_train_ctu, X_test_ctu, y_train_ctu, y_test_ctu = train_test_split(
    X_ctu, y_ctu, test_size=0.2, stratify=y_ctu, random_state=42)
df_model = df_model[['saddr', 'daddr', 'scenario', 'Label']]       # keep only what R2-R5 need
gc.collect()

# ---- 5) base models (LightGBM = D, XGBoost = C) ----
ctu_model = LGBMClassifier(n_estimators=200, random_state=SEED, n_jobs=-1, verbose=-1).fit(X_train_ctu, y_train_ctu)
xgb_ctu = XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric='logloss', tree_method='hist').fit(X_train_ctu, y_train_ctu)

# sanity check: should be close to the paper's base model (F1 0.827, recall 0.775, precision ~0.887)
yp = ctu_model.predict(X_test_ctu); pr = ctu_model.predict_proba(X_test_ctu)[:, 1]
print('\nLightGBM base on the test split ->',
      'P=%.4f R=%.4f F1=%.4f AUC=%.4f' % (precision_score(y_test_ctu, yp), recall_score(y_test_ctu, yp),
                                            f1_score(y_test_ctu, yp), roc_auc_score(y_test_ctu, pr)))
print('train/test rows:', len(X_train_ctu), len(X_test_ctu), '| train attacks:', int(y_train_ctu.sum()))


  scenario 42: 864637 rows kept
  scenario 43: 548123 rows kept
  scenario 44: 1413192 rows kept

=== DATA ATTRITION AUDIT ===
          rows_loaded  attacks_loaded  rows_modelled  attacks_modelled  \
scenario                                                                 
42             864637           12191         801813              8388   
43             548123            6267         464331              2226   
44            1413192            8011        1249902               789   

          attacks_dropped  pct_attacks_dropped  
scenario                                        
42                   3803                 31.2  
43                   4041                 64.5  
44                   7222                 90.2  

Missing values in the dropped ATTACK rows, by column:
dur           0
spkts         0
dpkts         0
sbytes        0
dbytes        0
rate          0
srate         0
drate         0
sttl          0
dttl      15066
dtype: int64

LightGBM base on the test sp

## R2 helpers

In [5]:
import ipaddress
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, precision_recall_curve)

def _valid_ip(ip):
    try:
        ipaddress.ip_address(str(ip)); return True
    except ValueError:
        return False

def geo_signals(frame):
    # G = 1 if source and destination countries differ AND both are geolocatable (Eq. 5)
    # E = 1 if both IP fields are present and valid IP addresses
    # Only unique IPs are looked up, so this is fast.
    uniq = pd.unique(pd.concat([frame['saddr'], frame['daddr']]))
    cmap = {ip: get_country(ip) for ip in uniq}
    vmap = {ip: _valid_ip(ip) for ip in uniq}
    s = frame['saddr'].map(cmap); d = frame['daddr'].map(cmap)
    ok = ~(s.str.startswith('Not Geolocatable') | d.str.startswith('Not Geolocatable') |
           (s == 'Unknown') | (d == 'Unknown'))
    G = ((s != d) & ok).astype(float).values
    E = (frame['saddr'].map(vmap) & frame['daddr'].map(vmap)).astype(float).values
    return G, E, s.values, d.values

def metrics(y, pred, score=None):
    out = {'Precision': precision_score(y, pred, zero_division=0),
           'Recall': recall_score(y, pred), 'F1': f1_score(y, pred)}
    if score is not None:
        out['ROC-AUC'] = roc_auc_score(y, score)
        out['PR-AUC'] = average_precision_score(y, score)
    return {k: round(float(v), 4) for k, v in out.items()}

def bootstrap_ci(y, pred, n_boot=500, seed=SEED):
    # seeded percentile bootstrap (Eq. 16) for Precision / Recall / F1
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(bool); pred = np.asarray(pred).astype(bool); n = len(y)
    P, R, F = [], [], []
    for _ in range(n_boot):
        i = rng.integers(0, n, n); yt, yp = y[i], pred[i]
        tp = (yt & yp).sum(); fp = (~yt & yp).sum(); fn = (yt & ~yp).sum()
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        P.append(p); R.append(r); F.append(2*p*r/(p + r) if p + r else 0.0)
    q = lambda a: (round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4))
    return {'Precision': q(P), 'Recall': q(R), 'F1': q(F)}

def mcnemar_exact(y, pred1, pred2):
    from scipy.stats import binomtest
    y = np.asarray(y); c1 = (np.asarray(pred1) == y); c2 = (np.asarray(pred2) == y)
    b = int((c1 & ~c2).sum()); c = int((~c1 & c2).sum())
    return (1.0 if b + c == 0 else float(binomtest(min(b, c), b + c, 0.5).pvalue)), b, c

def best_f1_threshold(y, score):
    p, r, t = precision_recall_curve(y, score)
    f = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[int(np.argmax(f))])
print('Helpers ready.')

Helpers ready.


## R3 fusion with validation-fitted weights/thresholds

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# 1) split the TRAINING part again into fit (75%) and validation (25%)
Xa, Xv, ya, yv = train_test_split(X_train_ctu, y_train_ctu, test_size=0.25,
                                  stratify=y_train_ctu, random_state=SEED)
lgb_a = LGBMClassifier(n_estimators=200, random_state=SEED, n_jobs=-1).fit(Xa, ya)
xgb_a = XGBClassifier(n_estimators=200, random_state=SEED, n_jobs=-1,
                      eval_metric='logloss', tree_method='hist').fit(Xa, ya)

def get_signals(X):
    C = xgb_a.predict_proba(X)[:, 1]            # confidence  (XGBoost)
    D = lgb_a.predict_proba(X)[:, 1]            # domain model (LightGBM)
    G, E, _, _ = geo_signals(df_model.loc[X.index])
    return C, G, D, E

Cv, Gv, Dv, Ev = get_signals(Xv)
Ct, Gt, Dt, Et = get_signals(X_test_ctu)

# 2) learned weights on validation data
stk = LogisticRegression(max_iter=1000, random_state=SEED).fit(np.c_[Cv, Gv, Dv, Ev], yv)
print('Learned coefficients  C, G, D, E =', np.round(stk.coef_[0], 3), '| intercept', np.round(stk.intercept_, 3))
agef_learned_v = stk.predict_proba(np.c_[Cv, Gv, Dv, Ev])[:, 1]
agef_learned_t = stk.predict_proba(np.c_[Ct, Gt, Dt, Et])[:, 1]

# 3) fixed (proposed) weights, same retrained models
w = dict(C=0.40, G=0.25, D=0.20, E=0.15)
agef_fixed_v = w['C']*Cv + w['G']*Gv + w['D']*Dv + w['E']*Ev
agef_fixed_t = w['C']*Ct + w['G']*Gt + w['D']*Dt + w['E']*Et

# 4) thresholds tuned on VALIDATION only (best F1), then applied once to the test set
systems = {
    'LightGBM alone (D), thr=0.5':            (Dv, Dt, 0.5),
    'LightGBM alone (D), tuned thr':          (Dv, Dt, None),
    'XGBoost alone (C), thr=0.5':             (Cv, Ct, 0.5),
    'XGBoost alone (C), tuned thr':           (Cv, Ct, None),
    'AGEF-Net fixed weights, thr=0.5':        (agef_fixed_v, agef_fixed_t, 0.5),
    'AGEF-Net fixed weights, tuned thr':      (agef_fixed_v, agef_fixed_t, None),
    'AGEF-Net LEARNED weights, tuned thr':    (agef_learned_v, agef_learned_t, None),
}
rows, preds = [], {}
for name, (sv, st, thr) in systems.items():
    thr = best_f1_threshold(yv, sv) if thr is None else thr
    pred = (st >= thr).astype(int); preds[name] = pred
    rows.append({'System': name, 'thr': round(thr, 3), **metrics(y_test_ctu, pred, st)})
fusion_table = pd.DataFrame(rows)
print(fusion_table.to_string(index=False))
fusion_table.to_csv('table_fusion_comparison_validation_tuned.csv', index=False)

# 5) is the learned fusion different from the best single model? (exact McNemar + seeded CIs)
best_single = fusion_table[fusion_table['System'].str.contains('alone')].sort_values('F1').iloc[-1]['System']
p, b, c = mcnemar_exact(y_test_ctu, preds[best_single], preds['AGEF-Net LEARNED weights, tuned thr'])
print(f'\nMcNemar: {best_single}  vs  learned AGEF-Net -> p={p:.4g} (b={b}, c={c})')
print('Bootstrap 95% CI, learned AGEF-Net:', bootstrap_ci(y_test_ctu, preds['AGEF-Net LEARNED weights, tuned thr']))
print('Bootstrap 95% CI, ', best_single, ':', bootstrap_ci(y_test_ctu, preds[best_single]))

Learned coefficients  C, G, D, E = [10.187 -0.078  5.489 -3.749] | intercept [-3.875]
                             System   thr  Precision  Recall     F1  ROC-AUC  PR-AUC
        LightGBM alone (D), thr=0.5 0.500     0.8653  0.7659 0.8126   0.9841  0.7919
      LightGBM alone (D), tuned thr 0.491     0.8654  0.7668 0.8131   0.9841  0.7919
         XGBoost alone (C), thr=0.5 0.500     0.9550  0.8558 0.9027   0.9991  0.9511
       XGBoost alone (C), tuned thr 0.478     0.9533  0.8597 0.9041   0.9991  0.9511
    AGEF-Net fixed weights, thr=0.5 0.500     0.8621  0.8663 0.8642   0.9501  0.8792
  AGEF-Net fixed weights, tuned thr 0.645     0.9556  0.8216 0.8835   0.9501  0.8792
AGEF-Net LEARNED weights, tuned thr 0.396     0.9532  0.8492 0.8982   0.9961  0.9478

McNemar: XGBoost alone (C), tuned thr  vs  learned AGEF-Net -> p=0.01058 (b=49, c=26)
Bootstrap 95% CI, learned AGEF-Net: {'Precision': (0.9435, 0.9622), 'Recall': (0.8346, 0.8644), 'F1': (0.8887, 0.9076)}
Bootstrap 95% CI,  XGBoost 

In [7]:
import shutil, glob, os
os.makedirs('/content/drive/MyDrive/agef_backup', exist_ok=True)
for f in glob.glob('/content/table_*.csv') + glob.glob('/content/shap_*.png'):
    shutil.copy(f, '/content/drive/MyDrive/agef_backup/')
print('Saved to Drive:', sorted(os.listdir('/content/drive/MyDrive/agef_backup')))

Saved to Drive: ['ctu13_attack.csv', 'ctu13_normal.csv', 'digital_evidence_report.json', 'digital_evidence_report_ctu13.json', 'evidence_hash_certificate.json', 'fig1_model_comparison.png', 'fig2_destination_countries.png', 'fig3_crossborder_pie.png', 'fig4_confusion_matrix.png', 'shap_summary.png', 'shap_summary_ctu13.png', 'table1_model_comparison.csv', 'table2_dataset_summary.csv', 'table_ctu13_model_comparison.csv', 'table_data_attrition_audit.csv', 'table_fusion_comparison_validation_tuned.csv', 'table_hard_attack_model_comparison.csv', 'table_per_attack_type_performance.csv']


## R4 threshold-matched baselines

In [8]:
C0 = xgb_ctu.predict_proba(X_test_ctu)[:, 1]
D0 = ctu_model.predict_proba(X_test_ctu)[:, 1]
G0, E0, _, _ = geo_signals(df_model.loc[X_test_ctu.index])
AGEF0 = 0.40*C0 + 0.25*G0 + 0.20*D0 + 0.15*E0
pred_agef0 = (AGEF0 >= 0.5).astype(int)
target_recall = recall_score(y_test_ctu, pred_agef0)

def precision_at_recall(y, score, target):
    p, r, t = precision_recall_curve(y, score)
    ok = np.where(r[:-1] >= target)[0]
    i = ok[np.argmax(p[:-1][ok])]
    return round(float(p[i]), 4), round(float(r[i]), 4), round(float(t[i]), 4)

print(f'AGEF-Net (fixed weights, thr 0.5): {metrics(y_test_ctu, pred_agef0, AGEF0)}')
for nm, sc in [('LightGBM alone (D)', D0), ('XGBoost alone (C)', C0)]:
    p, r, t = precision_at_recall(y_test_ctu, sc, target_recall)
    print(f'{nm} forced to recall >= {target_recall:.4f}: precision={p}, recall={r}, threshold={t}, PR-AUC={average_precision_score(y_test_ctu, sc):.4f}, ROC-AUC={roc_auc_score(y_test_ctu, sc):.4f}')
print(f'AGEF-Net PR-AUC={average_precision_score(y_test_ctu, AGEF0):.4f}, ROC-AUC={roc_auc_score(y_test_ctu, AGEF0):.4f}')
print('\nIf a single model matches AGEF-Net recall with HIGHER precision, say so in the paper: the gain is a threshold effect, not fusion.')

AGEF-Net (fixed weights, thr 0.5): {'Precision': 0.8713, 'Recall': 0.8575, 'F1': 0.8643, 'ROC-AUC': 0.951, 'PR-AUC': 0.876}
LightGBM alone (D) forced to recall >= 0.8575: precision=0.7386, recall=0.8584, threshold=0.0788, PR-AUC=0.8386, ROC-AUC=0.9887
XGBoost alone (C) forced to recall >= 0.8575: precision=0.9418, recall=0.858, threshold=0.4512, PR-AUC=0.9381, ROC-AUC=0.9977
AGEF-Net PR-AUC=0.8760, ROC-AUC=0.9510

If a single model matches AGEF-Net recall with HIGHER precision, say so in the paper: the gain is a threshold effect, not fusion.


## R5 leave-one-scenario-out

In [9]:
rows = []
for held in sorted(df_model['scenario'].unique()):
    tr = df_model['scenario'] != held
    Xtr, ytr = X_ctu[tr.values], y_ctu[tr.values]
    Xte, yte = X_ctu[~tr.values], y_ctu[~tr.values]
    if yte.sum() == 0 or ytr.sum() == 0:
        continue
    m_l = LGBMClassifier(n_estimators=200, random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
    m_x = XGBClassifier(n_estimators=200, random_state=SEED, n_jobs=-1, eval_metric='logloss', tree_method='hist').fit(Xtr, ytr)
    Cx, Dx = m_x.predict_proba(Xte)[:, 1], m_l.predict_proba(Xte)[:, 1]
    Gx, Ex, _, _ = geo_signals(df_model.loc[Xte.index])
    Ax = 0.40*Cx + 0.25*Gx + 0.20*Dx + 0.15*Ex
    for nm, sc in [('LightGBM alone', Dx), ('XGBoost alone', Cx), ('AGEF-Net (fixed weights)', Ax)]:
        rows.append({'Held-out scenario': held, 'System': nm, **metrics(yte, (sc >= 0.5).astype(int), sc)})
loso = pd.DataFrame(rows)
print(loso.to_string(index=False))
loso.to_csv('table_leave_one_scenario_out.csv', index=False)
print('\nNote: scenarios 42/43 are Neris, 44 is Rbot - holding out 44 tests a new botnet family.')

 Held-out scenario                   System  Precision  Recall     F1  ROC-AUC  PR-AUC
                42           LightGBM alone     0.2532  0.0374 0.0652   0.9679  0.2643
                42            XGBoost alone     0.5813  0.0349 0.0659   0.6224  0.0791
                42 AGEF-Net (fixed weights)     0.3148  0.0433 0.0761   0.8293  0.2513
                43           LightGBM alone     0.2633  0.1069 0.1521   0.9516  0.1559
                43            XGBoost alone     0.6381  0.1671 0.2649   0.9721  0.3727
                43 AGEF-Net (fixed weights)     0.5270  0.3288 0.4050   0.9411  0.4618
                44           LightGBM alone     0.5948  0.7554 0.6655   0.9705  0.4016
                44            XGBoost alone     0.5919  0.6286 0.6097   0.9704  0.5049
                44 AGEF-Net (fixed weights)     0.4877  0.9328 0.6406   0.9921  0.7092

Note: scenarios 42/43 are Neris, 44 is Rbot - holding out 44 tests a new botnet family.


In [10]:
import shutil, glob, os
os.makedirs('/content/drive/MyDrive/agef_backup', exist_ok=True)
for f in glob.glob('/content/table_*.csv') + glob.glob('/content/shap_*.png'):
    shutil.copy(f, '/content/drive/MyDrive/agef_backup/')
print('Saved to Drive:', sorted(os.listdir('/content/drive/MyDrive/agef_backup')))

Saved to Drive: ['ctu13_attack.csv', 'ctu13_normal.csv', 'digital_evidence_report.json', 'digital_evidence_report_ctu13.json', 'evidence_hash_certificate.json', 'fig1_model_comparison.png', 'fig2_destination_countries.png', 'fig3_crossborder_pie.png', 'fig4_confusion_matrix.png', 'shap_summary.png', 'shap_summary_ctu13.png', 'table1_model_comparison.csv', 'table2_dataset_summary.csv', 'table_ctu13_model_comparison.csv', 'table_data_attrition_audit.csv', 'table_fusion_comparison_validation_tuned.csv', 'table_hard_attack_model_comparison.csv', 'table_leave_one_scenario_out.csv', 'table_per_attack_type_performance.csv']
